# Moses (phrase-based SMT) trên dữ liệu song ngữ từng dòng

CPU, Internet bật. Không chứa test, không gắn cuộc thi.

In [ ]:
# ===== Cấu hình =====
CAU_HINH = {'huong': ['vi-zh'], 'seg': ['char'], 'mert_lap': 10, 'bien_the': ['train', 'khop4m_s42'], 'eval_cum': True}

In [ ]:
# Ghi mã vào working
from pathlib import Path
Path('kaggle_moses.py').write_text('# -*- coding: utf-8 -*-\n"""Moses (dich may thong ke theo cum tu) tren du lieu song ngu tung dong cua minh. Notebook Kaggle CPU, Internet BAT (apt + pip), khong test.\n\nQuy trinh (theo config/moses_pipeline.sh cua Quang, chon them tham so hop ly):\n  tach tu (vi: tach dau cau, giu so thap phan; zh: "char" = moi chu Han la mot token nhu Quang, hoac "jieba" = tach tu) -> loc cap (<=80 token, ti le <=9)\n  -> can chinh eflomal (model 3, 4-4-4 lan lap) 2 chieu -> doi xung grow-diag-final-and (viet bang Python) -> mo hinh ngon ngu KenLM 5-gram (Quang: 3-gram)\n  -> train-model.perl (buoc 4-9, cum toi da 7 tu, reordering msd-bidirectional-fe) -> MERT tren dev 1.000 dong (200 title + 800 dong thuoc tinh, fold 0, ngoai mau eval)\n  -> loc bang cum theo dau vao -> giai ma eval (cung 1.200 mau eval cua LLM, tung dong) -> ghep lai chu -> sinh.jsonl {id, nhom, hyp}.\nMoses duoc bien dich san o notebook nlp-moses-build (dau ra moses_bin_scripts.tgz gan qua kernel_sources). Moi (huong, seg) la mot mo hinh;\nchon seg theo chrF tren DEV (khong phai eval), bao cao ca hai.\n"""\nimport json\nimport math\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nVAO = Path("/kaggle/input")\nRA = Path("/kaggle/working")\nMOSES = Path("/tmp/moses")\nB, S = MOSES / "bin", MOSES / "scripts"\n\n# ---------- tach tu / ghep lai (ham thuan, co test) ----------\nRE_VI = re.compile(r"\\d+(?:[.,]\\d+)*|\\w+|[^\\w\\s]", re.UNICODE)\nRE_ZH_CHAR = re.compile(r"[A-Za-z0-9]+(?:[.,][0-9]+)*|[㐀-鿿]|[^\\sA-Za-z0-9㐀-鿿]")\n_jieba = None\n\n\ndef tok_vi(s):\n    return RE_VI.findall(s)\n\n\ndef tok_zh_char(s):\n    return RE_ZH_CHAR.findall(s)\n\n\ndef tok_zh_jieba(s):\n    global _jieba\n    if _jieba is None:\n        import jieba\n        jieba.setLogLevel(60)\n        _jieba = jieba\n    return [t.strip() for t in _jieba.lcut(s) if t.strip()]\n\n\ndef lay_tach(ngon_ngu, seg):\n    if ngon_ngu == "vi":\n        return tok_vi\n    return tok_zh_char if seg == "char" else tok_zh_jieba\n\n\ndef detok_vi(toks):\n    s = " ".join(toks)\n    s = re.sub(r"\\s+([,.;:!?%)\\]}])", r"\\1", s)\n    s = re.sub(r"([(\\[{])\\s+", r"\\1", s)\n    s = re.sub(r"(?<=\\w)\\s*([/-])\\s*(?=\\w)", r"\\1", s)\n    return s.strip()\n\n\ndef detok_zh(toks):\n    out = ""\n    for t in toks:\n        if out and re.match(r"[A-Za-z0-9]", out[-1]) and re.match(r"[A-Za-z0-9]", t[0]):\n            out += " "\n        out += t\n    return out.strip()\n\n\ndef ghep(ngon_ngu, dong):\n    toks = dong.split()\n    return detok_vi(toks) if ngon_ngu == "vi" else detok_zh(toks)\n\n\n# ---------- doi xung hoa can chinh: grow-diag-final-and ----------\ndef doc_cap(dong):\n    return {tuple(int(x) for x in p.split("-")) for p in dong.split()}\n\n\ndef grow_diag_final_and(fwd, rev):\n    """fwd, rev: tap (i, j) = (chi so nguon, chi so dich). Tra ve tap cap sau grow-diag-final-and (Koehn et al. 2003)."""\n    union = fwd | rev\n    A = set(fwd & rev)\n    ns = {i for i, _ in A}\n    nt = {j for _, j in A}\n    lang_gieng = [(-1, 0), (0, -1), (1, 0), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]\n    doi = True\n    while doi:\n        doi = False\n        for (i, j) in sorted(A):\n            for di, dj in lang_gieng:\n                p = (i + di, j + dj)\n                if p in union and p not in A and (p[0] not in ns or p[1] not in nt):\n                    A.add(p)\n                    ns.add(p[0])\n                    nt.add(p[1])\n                    doi = True\n    for p in sorted(union):              # final-and: them diem noi ca hai dau deu chua co can chinh\n        if p not in A and p[0] not in ns and p[1] not in nt:\n            A.add(p)\n            ns.add(p[0])\n            nt.add(p[1])\n    return A\n\n\ndef doi_xung(tep_fwd, tep_rev, tep_ra, n_nguon, n_dich):\n    """Doc hai tep \'i-j\' (moi dong mot cau), ghi tep doi xung. Tu kiem huong tep rev (neu chi so vuot do dai thi dao)."""\n    lat = None\n    n = 0\n    with open(tep_fwd, encoding="utf-8") as f, open(tep_rev, encoding="utf-8") as r, open(tep_ra, "w", encoding="utf-8") as o:\n        for k, (a, b) in enumerate(zip(f, r)):\n            fa, rb = doc_cap(a), doc_cap(b)\n            if lat is None and k < 2000:\n                sai = sum(1 for (i, j) in rb if i >= n_nguon[k] or j >= n_dich[k])\n                sai_dao = sum(1 for (i, j) in rb if j >= n_nguon[k] or i >= n_dich[k])\n                if sai > 0 and sai_dao == 0:\n                    lat = True\n            if lat:\n                rb = {(j, i) for (i, j) in rb}\n            o.write(" ".join(f"{i}-{j}" for i, j in sorted(grow_diag_final_and(fa, rb))) + "\\n")\n            n += 1\n    return n, bool(lat)\n\n\n# ---------- he thong ----------\ndef sh(cmd, tail=1500, check=True, log=None):\n    t = time.time()\n    r = subprocess.run(["bash", "-lc", "set -o pipefail; " + cmd], text=True, capture_output=True)\n    out = (r.stdout or "") + (r.stderr or "")\n    if log:\n        Path(log).write_text(out, encoding="utf-8")\n    print(">>", cmd[:260], "| ma", r.returncode, "| %.0fs" % (time.time() - t), flush=True)\n    print(out[-tail:], flush=True)\n    if check and r.returncode:\n        raise RuntimeError(f"lenh_loi:{r.returncode}:{cmd[:140]}")\n    return r.returncode, out\n\n\ndef tim(ten):\n    ds = [p for p in VAO.rglob(ten) if p.is_file()]\n    if not ds:\n        raise RuntimeError(f"khong_thay:{ten}")\n    return ds[0]\n\n\ndef doc(ten, toi_da=None):\n    L = tim(ten).read_text(encoding="utf-8").split("\\n")\n    if L and L[-1] == "":\n        L = L[:-1]\n    return L[:toi_da] if toi_da else L\n\n\ndef cai():\n    sh("apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq libboost-all-dev zlib1g-dev libbz2-dev liblzma-dev libgoogle-perftools-dev libsparsehash-dev > /tmp/apt.log 2>&1; tail -2 /tmp/apt.log")\n    tgz = [p for p in VAO.rglob("moses_bin_scripts.tgz")]\n    if not tgz:\n        raise RuntimeError("khong_thay_moses_bin_scripts.tgz (gan notebook nlp-moses-build qua kernel_sources)")\n    MOSES.mkdir(parents=True, exist_ok=True)\n    sh(f"tar xzf {tgz[0]} -C {MOSES} && chmod -R +x {B} && ls {B} | wc -l && {B}/moses --version 2>&1 | head -2")\n    sh("pip install -q eflomal jieba sacrebleu 2>&1 | tail -3")\n\n\ndef chrf_dev(hyp, ref, ngon_ngu):\n    from sacrebleu.metrics import CHRF\n    return CHRF(word_order=2 if ngon_ngu == "vi" else 0).corpus_score(hyp, [ref]).score\n\n\ndef chay_mot(hg, seg, bt, cfg, W0):\n    sl, tl = hg.split("-")\n    t0 = time.time()\n    k = cfg.get("khoi")\n    W = W0 / f"{hg}_{seg}_{bt}"\n    W.mkdir(parents=True, exist_ok=True)\n    tok_s, tok_t = lay_tach(sl, seg), lay_tach(tl, seg)\n    tr_s, tr_t = doc(f"{hg}.{bt}.src", k), doc(f"{hg}.{bt}.tgt", k)\n    dv_s, dv_t = doc(f"{hg}.dev.src", 200 if k else None), doc(f"{hg}.dev.tgt", 200 if k else None)\n    ev_s, ev_id, ev_nh = doc(f"{hg}.eval.src", k), doc(f"{hg}.eval.id", k), doc(f"{hg}.eval.nhom", k)\n    # 1) tach tu + loc cap\n    A, Bt, ns, nt = [], [], [], []\n    for s, t in zip(tr_s, tr_t):\n        a, b = tok_s(s), tok_t(t)\n        if 1 <= len(a) <= 80 and 1 <= len(b) <= 80 and max(len(a), len(b)) / max(1, min(len(a), len(b))) <= 9:\n            A.append(" ".join(a)); Bt.append(" ".join(b)); ns.append(len(a)); nt.append(len(b))\n    (W / f"train.{sl}").write_text("\\n".join(A) + "\\n", encoding="utf-8")\n    (W / f"train.{tl}").write_text("\\n".join(Bt) + "\\n", encoding="utf-8")\n    (W / f"dev.{sl}").write_text("\\n".join(" ".join(tok_s(x)) for x in dv_s) + "\\n", encoding="utf-8")\n    (W / f"dev.{tl}").write_text("\\n".join(" ".join(tok_t(x)) for x in dv_t) + "\\n", encoding="utf-8")\n    (W / f"eval.{sl}").write_text("\\n".join(" ".join(tok_s(x)) for x in ev_s) + "\\n", encoding="utf-8")\n    # eval ban cum (tung manh) dich cung luot: ghi chung vao evalall de loc bang cum mot lan\n    cum_s = doc(f"{hg}.evalcum.src", k) if cfg.get("eval_cum") else []\n    cum_id = doc(f"{hg}.evalcum.id", k) if cfg.get("eval_cum") else []\n    (W / f"evalall.{sl}").write_text("\\n".join(" ".join(tok_s(x)) for x in ev_s + cum_s) + "\\n", encoding="utf-8")\n    print(f"[{hg}/{seg}] cap huan luyen {len(tr_s)} -> {len(A)} sau loc; token nguon tb {sum(ns)/len(ns):.1f}, dich tb {sum(nt)/len(nt):.1f}; {time.time()-t0:.0f}s", flush=True)\n    # 2) can chinh eflomal 2 chieu + doi xung\n    from eflomal import Aligner\n    al = Aligner(model=3, n_iterations=(4, 4, 4), n_samplers=4)\n    with open(W / f"train.{sl}", encoding="utf-8") as fs, open(W / f"train.{tl}", encoding="utf-8") as ft:\n        al.align(fs, ft, links_filename_fwd=str(W / "fwd"), links_filename_rev=str(W / "rev"), quiet=True)\n    n_cap, da_lat = doi_xung(W / "fwd", W / "rev", W / "sym.grow-diag-final-and", ns, nt)\n    print(f"[{hg}/{seg}] can chinh xong {n_cap} cau (dao huong rev: {da_lat}); {time.time()-t0:.0f}s", flush=True)\n    # 3) mo hinh ngon ngu 5-gram KenLM (chi tren phia dich cua train)\n    (W / "lmtmp").mkdir(exist_ok=True)\n    sh(f"{B}/lmplz -o 5 -S 8G -T {W}/lmtmp --discount_fallback < {W}/train.{tl} > {W}/lm.arpa && {B}/build_binary trie {W}/lm.arpa {W}/lm.bin && rm -f {W}/lm.arpa", tail=600)\n    # 4) bang cum + reordering\n    (W / "tm" / "model").mkdir(parents=True, exist_ok=True)       # bat dau tu buoc 4 thi script khong tu tao thu muc\n    sh(f"perl {S}/training/train-model.perl -root-dir {W}/tm -corpus {W}/train -f {sl} -e {tl} -alignment grow-diag-final-and "\n       f"-alignment-file {W}/sym -first-step 4 -last-step 9 -lm 0:5:{W}/lm.bin:8 -max-phrase-length 7 -reordering msd-bidirectional-fe "\n       f"-cores 4 -sort-buffer-size 4G", tail=2500, log=W / "train_model.log")\n    ini = W / "tm" / "model" / "moses.ini"\n    # 5) MERT tren dev\n    tuned = False\n    try:\n        sh(f"perl {S}/training/mert-moses.pl {W}/dev.{sl} {W}/dev.{tl} {B}/moses {ini} --working-dir {W}/mert --mertdir {B} --rootdir {S} "\n           f"--decoder-flags=\'-threads 4 -v 0\' --nbest 100 --maximum-iterations {cfg.get(\'mert_lap\', 10)}", tail=1500, log=W / "mert.log")\n        if (W / "mert" / "moses.ini").exists():\n            ini, tuned = W / "mert" / "moses.ini", True\n    except Exception as e:\n        print("MERT loi, dung trong so mac dinh:", e, flush=True)\n    # 6) giai ma dev (de chon seg) va eval, deu qua loc bang cum theo dau vao\n    ket = {}\n    for ten, n_ in (("dev", None), ("evalall", None)):\n        sh(f"perl {S}/training/filter-model-given-input.pl {W}/f_{ten} {ini} {W}/{ten}.{sl} > /dev/null 2>&1; ls {W}/f_{ten}/moses.ini", tail=200)\n        sh(f"{B}/moses -f {W}/f_{ten}/moses.ini -threads 4 -v 0 < {W}/{ten}.{sl} > {W}/{ten}.hyp.tok", tail=300, log=W / f"giai_ma_{ten}.log")\n    dev_hyp = [ghep(tl, x) for x in (W / "dev.hyp.tok").read_text(encoding="utf-8").split("\\n")[:len(dv_t)]]\n    dev_chrf = round(chrf_dev(dev_hyp, dv_t, tl), 2)\n    tat_ca = [ghep(tl, x) for x in (W / "evalall.hyp.tok").read_text(encoding="utf-8").split("\\n")[:len(ev_s) + len(cum_s)]]\n    ev_hyp, cum_hyp = tat_ca[:len(ev_s)], tat_ca[len(ev_s):]\n    ra = RA / "moses" / f"{hg}_{seg}_{bt}"\n    ra.mkdir(parents=True, exist_ok=True)\n    (ra / "sinh.jsonl").write_text("\\n".join(json.dumps({"id": i, "nhom": g, "hyp": h}, ensure_ascii=False) for i, g, h in zip(ev_id, ev_nh, ev_hyp)), encoding="utf-8")\n    n_cum = 0\n    if cum_s:      # ghep cac manh cua cung mot mau cum bang "; " (title chi co mot manh)\n        gom = {}\n        for i, h in zip(cum_id, cum_hyp):\n            gom.setdefault(i.rsplit("#", 1)[0], []).append(h)\n        nhom_cum = {json.loads(l)["id"]: json.loads(l)["nhom"] for l in tim(f"{hg}.evalcum_items.jsonl").read_text(encoding="utf-8").split("\\n") if l.strip()}\n        (ra / "sinh_cum.jsonl").write_text("\\n".join(json.dumps({"id": i, "nhom": nhom_cum.get(i, ""), "hyp": "; ".join(v)}, ensure_ascii=False) for i, v in gom.items()), encoding="utf-8")\n        n_cum = len(gom)\n    ve = {"huong": hg, "seg": seg, "bien_the": bt, "n_eval_cum": n_cum, "n_manh_cum": len(cum_s), "dong_train": len(tr_s), "dong_sau_loc": len(A), "mert": tuned, "dev_chrf": dev_chrf, "n_eval": len(ev_s),\n          "so_hyp_rong": sum(1 for x in ev_hyp if not x.strip()), "giay": round(time.time() - t0, 1)}\n    sh(f"cp {W}/train_model.log {W}/mert.log {ra}/ 2>/dev/null; cp {ini} {ra}/moses_tuned.ini 2>/dev/null; rm -rf {W}", check=False, tail=100)\n    return ve\n\n\ndef main(cfg):\n    t0 = time.time()\n    cai()\n    W0 = Path("/tmp/mw")\n    W0.mkdir(exist_ok=True)\n    bc = {"cau_hinh": cfg, "viec": []}\n    for hg in cfg["huong"]:\n        for seg in cfg["seg"]:\n            for bt in cfg.get("bien_the", ["train"]):\n                try:\n                    bc["viec"].append(chay_mot(hg, seg, bt, cfg, W0))\n                except Exception as e:\n                    import traceback\n                    bc["viec"].append({"huong": hg, "seg": seg, "bien_the": bt, "loi": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc()[-2500:]})\n                    print("LOI", hg, seg, bt, e, flush=True)\n                (RA / "moses_report.json").write_text(json.dumps(bc, ensure_ascii=False, indent=1), encoding="utf-8")\n    bc["giay_tong"] = round(time.time() - t0, 1)\n    (RA / "moses_report.json").write_text(json.dumps(bc, ensure_ascii=False, indent=1), encoding="utf-8")\n    print("XONG", len(bc["viec"]), flush=True)\n\n\nif __name__ == "__main__":\n    raise SystemExit("Nhung vao notebook, goi main(CAU_HINH)")\n\n', encoding='utf-8')


In [ ]:
import sys
sys.path.insert(0, '.')
import kaggle_moses
kaggle_moses.main(CAU_HINH)
